# SIT720 9.1P — Decision Tree and Ensemble Methods

Complete practical notebook for the UCI **Car Evaluation Dataset**.

It covers:
1. data exploration and preprocessing,
2. full Decision Tree,
3. tree interpretation and root impurity reduction,
4. tree depth / bias–variance experiment,
5. tuned Random Forest with OOB error and feature importance,
6. tuned AdaBoost,
7. final model comparison.

Run all cells from top to bottom and use the actual outputs when writing your report.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", None)

## Part B1 — Data Exploration and Pre-processing

In [2]:
# =========================
# 1.1 Load dataset
# =========================
from pathlib import Path
columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
local_path = Path("car.data")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
if local_path.exists():
    df = pd.read_csv(local_path, header=None, names=columns)
    print("Dataset loaded from local car.data file.")
else:
    df = pd.read_csv(url, header=None, names=columns)
    print("Dataset loaded from UCI.")
display(df.head())
print("Shape:", df.shape)

Dataset loaded from local car.data file.


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


Shape: (1728, 7)


In [3]:
target_map = {
    "unacc": "Unacceptable",
    "acc": "Acceptable",
    "good": "Good",
    "vgood": "Good"
}
df["target"] = df["class"].map(target_map)

feature_cols = ["buying", "maint", "doors", "persons", "lug_boot", "safety"]

print("Observations:", len(df))
print("Input features:", len(feature_cols))
print("\nData types:")
display(df[feature_cols + ["target"]].dtypes.to_frame("dtype"))
print("\nMissing values:")
display(df[feature_cols + ["target"]].isna().sum().to_frame("missing"))
print("\nOriginal class distribution:")
display(df["class"].value_counts().to_frame("count"))
print("\nThree-class distribution:")
display(df["target"].value_counts().to_frame("count"))
print("\nThree-class percentages:")
display((df["target"].value_counts(normalize=True)*100).round(2).to_frame("percent"))

Observations: 1728
Input features: 6

Data types:


,dtype
buying,object
maint,object
doors,object
persons,object
lug_boot,object
safety,object
target,object



Missing values:


,missing
buying,0
maint,0
doors,0
persons,0
lug_boot,0
safety,0
target,0



Original class distribution:


,count
class,
unacc,1210
acc,384
good,69
vgood,65



Three-class distribution:


,count
target,
Unacceptable,1210
Acceptable,384
Good,134



Three-class percentages:


,percent
target,
Unacceptable,70.02
Acceptable,22.22
Good,7.75


In [4]:
class_order = ["Unacceptable", "Acceptable", "Good"]
plt.figure(figsize=(7,4))
df["target"].value_counts().reindex(class_order).plot(kind="bar")
plt.title("Car Evaluation Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Encoding justification
The six predictors are categorical but each has a meaningful order. This notebook therefore uses **ordinal encoding with explicit semantic category order** rather than arbitrary alphabetical encoding. The encoder is fitted on the training set only.

In [5]:
X = df[feature_cols].copy()
y = df["target"].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training shape:", X_train_raw.shape)
print("Testing shape:", X_test_raw.shape)
print("\nTraining distribution (%):")
display((y_train.value_counts(normalize=True)*100).round(2).to_frame("percent"))
print("\nTesting distribution (%):")
display((y_test.value_counts(normalize=True)*100).round(2).to_frame("percent"))

Training shape: (1382, 6)
Testing shape: (346, 6)

Training distribution (%):


,percent
target,
Unacceptable,70.04
Acceptable,22.21
Good,7.74



Testing distribution (%):


,percent
target,
Unacceptable,69.94
Acceptable,22.25
Good,7.80


In [6]:
category_order = [
    ["low", "med", "high", "vhigh"],
    ["low", "med", "high", "vhigh"],
    ["2", "3", "4", "5more"],
    ["2", "4", "more"],
    ["small", "med", "big"],
    ["low", "med", "high"]
]

encoder = OrdinalEncoder(
    categories=category_order,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)
X_train = encoder.fit_transform(X_train_raw)
X_test = encoder.transform(X_test_raw)

X_train_df = pd.DataFrame(X_train, columns=feature_cols, index=X_train_raw.index)
X_test_df = pd.DataFrame(X_test, columns=feature_cols, index=X_test_raw.index)
display(X_train_df.head())

,buying,maint,doors,persons,lug_boot,safety
1213,1.0,0.0,0.0,2.0,2.0,1.0
805,2.0,0.0,1.0,2.0,1.0,1.0
459,2.0,3.0,1.0,0.0,0.0,0.0
972,1.0,2.0,0.0,0.0,0.0,0.0
1020,1.0,2.0,1.0,2.0,1.0,0.0


In [7]:
labels = ["Unacceptable", "Acceptable", "Good"]

def evaluate_model(name, model, X_eval, y_eval, show_cm=True):
    pred = model.predict(X_eval)
    result = {
        "Model": name,
        "Accuracy": accuracy_score(y_eval, pred),
        "Precision_macro": precision_score(y_eval, pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_eval, pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_eval, pred, average="macro", zero_division=0),
        "F1_weighted": f1_score(y_eval, pred, average="weighted", zero_division=0)
    }
    print("\n" + "="*70)
    print(name)
    print("="*70)
    display(pd.DataFrame([result]).set_index("Model").round(4))
    print(classification_report(y_eval, pred, labels=labels, digits=4, zero_division=0))
    if show_cm:
        cm = confusion_matrix(y_eval, pred, labels=labels)
        ConfusionMatrixDisplay(cm, display_labels=labels).plot()
        plt.title(name + " — Confusion Matrix")
        plt.tight_layout()
        plt.show()
    return result, pred

## Part B2 — Baseline Full Decision Tree

In [8]:
full_tree = DecisionTreeClassifier(criterion="gini", random_state=RANDOM_STATE)
full_tree.fit(X_train, y_train)

baseline_results, baseline_pred = evaluate_model(
    "Full Decision Tree", full_tree, X_test, y_test
)
print("Tree depth:", full_tree.get_depth())
print("Number of leaves:", full_tree.get_n_leaves())


Full Decision Tree


,Accuracy,Precision_macro,Recall_macro,F1_macro,F1_weighted
Model,,,,,
Full Decision Tree,0.9913,0.9718,0.979,0.9753,0.9914


              precision    recall  f1-score   support

Unacceptable     1.0000    1.0000    1.0000       242
  Acceptable     0.9868    0.9740    0.9804        77
        Good     0.9286    0.9630    0.9455        27

    accuracy                         0.9913       346
   macro avg     0.9718    0.9790    0.9753       346
weighted avg     0.9915    0.9913    0.9914       346



Tree depth: 12
Number of leaves: 72


## Part B3 — Decision Tree Interpretation

In [9]:
plt.figure(figsize=(30,16))
plot_tree(
    full_tree,
    feature_names=feature_cols,
    class_names=full_tree.classes_,
    filled=True,
    rounded=True,
    fontsize=6
)
plt.title("Full Decision Tree")
plt.tight_layout()
plt.show()

plt.figure(figsize=(18,8))
plot_tree(
    full_tree,
    feature_names=feature_cols,
    class_names=full_tree.classes_,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=9
)
plt.title("Decision Tree — First Three Levels")
plt.tight_layout()
plt.show()

In [10]:
t = full_tree.tree_
root = 0
left = t.children_left[root]
right = t.children_right[root]
root_feature_idx = t.feature[root]
root_feature = feature_cols[root_feature_idx]
root_threshold = t.threshold[root]

print("Root feature:", root_feature)
print("First split threshold:", round(root_threshold, 4))
print(f"Encoded first split: {root_feature} <= {root_threshold:.4f}")

n_root = t.weighted_n_node_samples[root]
n_left = t.weighted_n_node_samples[left]
n_right = t.weighted_n_node_samples[right]
root_impurity = t.impurity[root]
left_impurity = t.impurity[left]
right_impurity = t.impurity[right]
weighted_child_impurity = (n_left/n_root)*left_impurity + (n_right/n_root)*right_impurity
reduction = root_impurity - weighted_child_impurity

print(f"Root Gini impurity: {root_impurity:.6f}")
print(f"Weighted child impurity: {weighted_child_impurity:.6f}")
print(f"Root Gini reduction: {reduction:.6f}")

Root feature: persons
First split threshold: 0.5
Encoded first split: persons <= 0.5000
Root Gini impurity: 0.454051
Weighted child impurity: 0.381667
Root Gini reduction: 0.072383


In [11]:
print(export_text(full_tree, feature_names=feature_cols, max_depth=4))

|--- persons <= 0.50
|   |--- class: Unacceptable
|--- persons >  0.50
|   |--- safety <= 0.50
|   |   |--- class: Unacceptable
|   |--- safety >  0.50
|   |   |--- buying <= 1.50
|   |   |   |--- maint <= 1.50
|   |   |   |   |--- safety <= 1.50
|   |   |   |   |   |--- truncated branch of depth 7
|   |   |   |   |--- safety >  1.50
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |--- maint >  1.50
|   |   |   |   |--- lug_boot <= 0.50
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |   |--- lug_boot >  0.50
|   |   |   |   |   |--- truncated branch of depth 8
|   |   |--- buying >  1.50
|   |   |   |--- maint <= 1.50
|   |   |   |   |--- lug_boot <= 0.50
|   |   |   |   |   |--- truncated branch of depth 4
|   |   |   |   |--- lug_boot >  0.50
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |--- maint >  1.50
|   |   |   |   |--- buying <= 2.50
|   |   |   |   |   |--- truncated branch of depth 8
|   |   |   |   |--- buying >  2.50
|

## Part B4 — Tree Complexity and Bias–Variance Trade-off

In [12]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
records = []

for depth in range(1, 16):
    model = DecisionTreeClassifier(
        criterion="gini", max_depth=depth, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    cv_scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1
    )
    records.append({
        "max_depth": depth,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "train_f1_macro": f1_score(y_train, train_pred, average="macro", zero_division=0),
        "test_f1_macro": f1_score(y_test, test_pred, average="macro", zero_division=0),
        "cv_f1_macro_mean": cv_scores.mean(),
        "cv_f1_macro_std": cv_scores.std()
    })

depth_df = pd.DataFrame(records)
display(depth_df.round(4))

,max_depth,train_accuracy,test_accuracy,train_f1_macro,test_f1_macro,cv_f1_macro_mean,cv_f1_macro_std
0,1,0.7004,0.6994,0.2746,0.2744,0.2746,0.0001
1,2,0.7728,0.7977,0.5140,0.5300,0.5140,0.0067
2,3,0.7873,0.8092,0.4761,0.5001,0.4760,0.0043
3,4,0.8799,0.8844,0.8141,0.7919,0.8139,0.0150
4,5,0.8929,0.8960,0.8106,0.8207,0.7950,0.0333
5,6,0.9392,0.9422,0.8931,0.8803,0.8904,0.0030
6,7,0.9465,0.9364,0.8984,0.8740,0.8314,0.0252
7,8,0.9805,0.9827,0.9617,0.9682,0.9186,0.0086
8,9,0.9863,0.9711,0.9756,0.9379,0.9208,0.0314
9,10,0.9964,0.9884,0.9936,0.9725,0.9378,0.0179


In [13]:
plt.figure(figsize=(9,5))
plt.plot(depth_df["max_depth"], depth_df["train_accuracy"], marker="o", label="Training accuracy")
plt.plot(depth_df["max_depth"], depth_df["test_accuracy"], marker="o", label="Testing accuracy")
plt.xlabel("Maximum depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree Accuracy vs Maximum Depth")
plt.xticks(range(1,16))
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,5))
plt.plot(depth_df["max_depth"], depth_df["train_f1_macro"], marker="o", label="Training macro-F1")
plt.plot(depth_df["max_depth"], depth_df["test_f1_macro"], marker="o", label="Testing macro-F1")
plt.plot(depth_df["max_depth"], depth_df["cv_f1_macro_mean"], marker="o", label="5-fold CV macro-F1")
plt.xlabel("Maximum depth")
plt.ylabel("Macro-F1")
plt.title("Decision Tree Macro-F1 vs Maximum Depth")
plt.xticks(range(1,16))
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [14]:
# Select depth using training-set cross-validation, not test results.
best_depth = int(depth_df.loc[depth_df["cv_f1_macro_mean"].idxmax(), "max_depth"])
print("Best depth by 5-fold training CV:", best_depth)

best_tree = DecisionTreeClassifier(
    criterion="gini", max_depth=best_depth, random_state=RANDOM_STATE
)
best_tree.fit(X_train, y_train)
best_tree_results, best_tree_pred = evaluate_model(
    f"Decision Tree (depth={best_depth})", best_tree, X_test, y_test
)

Best depth by 5-fold training CV: 10

Decision Tree (depth=10)


,Accuracy,Precision_macro,Recall_macro,F1_macro,F1_weighted
Model,,,,,
Decision Tree (depth=10),0.9884,0.9675,0.9776,0.9725,0.9885


              precision    recall  f1-score   support

Unacceptable     1.0000    0.9959    0.9979       242
  Acceptable     0.9740    0.9740    0.9740        77
        Good     0.9286    0.9630    0.9455        27

    accuracy                         0.9884       346
   macro avg     0.9675    0.9776    0.9725       346
weighted avg     0.9886    0.9884    0.9885       346



## Part B5 — Random Forest: Bagging Ensemble Learning

In [15]:
rf = RandomForestClassifier(
    random_state=RANDOM_STATE, bootstrap=True, n_jobs=-1
)
rf_grid_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10],
    "max_features": ["sqrt", None]
}

rf_grid = GridSearchCV(
    rf,
    rf_grid_params,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
rf_grid.fit(X_train, y_train)

print("Best Random Forest parameters:", rf_grid.best_params_)
print("Best CV macro-F1:", round(rf_grid.best_score_, 4))

Best Random Forest parameters: {'max_depth': 10, 'max_features': None, 'n_estimators': 100}
Best CV macro-F1: 0.9565


In [16]:
best_rf = RandomForestClassifier(
    **rf_grid.best_params_,
    random_state=RANDOM_STATE,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1
)
best_rf.fit(X_train, y_train)

rf_results, rf_pred = evaluate_model(
    "Optimised Random Forest", best_rf, X_test, y_test
)
print("OOB score:", round(best_rf.oob_score_, 4))
print("OOB error:", round(1-best_rf.oob_score_, 4))


Optimised Random Forest


,Accuracy,Precision_macro,Recall_macro,F1_macro,F1_weighted
Model,,,,,
Optimised Random Forest,0.9884,0.9675,0.9776,0.9725,0.9885


              precision    recall  f1-score   support

Unacceptable     1.0000    0.9959    0.9979       242
  Acceptable     0.9740    0.9740    0.9740        77
        Good     0.9286    0.9630    0.9455        27

    accuracy                         0.9884       346
   macro avg     0.9675    0.9776    0.9725       346
weighted avg     0.9886    0.9884    0.9885       346

OOB score: 0.9834
OOB error: 0.0166


In [17]:
importance = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
display(importance.to_frame("importance"))

plt.figure(figsize=(8,5))
importance.head(5).sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("Random Forest — Top Five Feature Importances")
plt.tight_layout()
plt.show()

Feature importances:


,importance
safety,0.295864
persons,0.214366
maint,0.200469
buying,0.132031
lug_boot,0.106156
doors,0.051114


## Part B6 — AdaBoost

In [18]:
stump = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)

try:
    ada = AdaBoostClassifier(estimator=stump, random_state=RANDOM_STATE)
except TypeError:
    ada = AdaBoostClassifier(base_estimator=stump, random_state=RANDOM_STATE)

ada_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.1, 1.0]
}

ada_grid = GridSearchCV(
    ada,
    ada_params,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
ada_grid.fit(X_train, y_train)

print("Best AdaBoost parameters:", ada_grid.best_params_)
print("Best CV macro-F1:", round(ada_grid.best_score_, 4))

Best AdaBoost parameters: {'learning_rate': 1.0, 'n_estimators': 200}
Best CV macro-F1: 0.8064


In [19]:
best_ada = ada_grid.best_estimator_
ada_results, ada_pred = evaluate_model(
    "Optimised AdaBoost", best_ada, X_test, y_test
)


Optimised AdaBoost


,Accuracy,Precision_macro,Recall_macro,F1_macro,F1_weighted
Model,,,,,
Optimised AdaBoost,0.8786,0.8412,0.8418,0.8348,0.8827


              precision    recall  f1-score   support

Unacceptable     0.9641    0.8884    0.9247       242
  Acceptable     0.6900    0.8961    0.7797        77
        Good     0.8696    0.7407    0.8000        27

    accuracy                         0.8786       346
   macro avg     0.8412    0.8418    0.8348       346
weighted avg     0.8957    0.8786    0.8827       346



## Final Comparison

In [20]:
comparison = pd.DataFrame([
    best_tree_results,
    rf_results,
    ada_results
]).set_index("Model")

display(comparison.round(4))

comparison[["Accuracy", "Precision_macro", "Recall_macro", "F1_macro"]].plot(
    kind="bar", figsize=(11,6)
)
plt.ylim(0,1.05)
plt.ylabel("Score")
plt.title("Decision Tree vs Random Forest vs AdaBoost")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

best_name = comparison["F1_macro"].idxmax()
print("Highest test macro-F1 model:", best_name)
print("Macro-F1:", round(comparison.loc[best_name, "F1_macro"], 4))

,Accuracy,Precision_macro,Recall_macro,F1_macro,F1_weighted
Model,,,,,
Decision Tree (depth=10),0.9884,0.9675,0.9776,0.9725,0.9885
Optimised Random Forest,0.9884,0.9675,0.9776,0.9725,0.9885
Optimised AdaBoost,0.8786,0.8412,0.8418,0.8348,0.8827


Highest test macro-F1 model: Decision Tree (depth=10)
Macro-F1: 0.9725


## GenAI acknowledgement template

Adapt this to match what you actually did:

> I used ChatGPT to assist with planning the notebook structure and debugging Python code. I executed the notebook, reviewed the outputs and wrote the final interpretation in my own words.